# TrustRAG C1 - ModernBERT span detector

Thin runner. All logic lives in `src/c1_detector/`; this notebook clones the repo,
installs what Kaggle is missing, points the code at the data and calls it. Anything
worth debugging should be debugged locally against `configs/c1_smoke.yaml`, not here.

## Before running - do these by hand, in this order

1. **Upload the data.** `data/processed/` is gitignored, so it does not arrive with
   the clone. On your machine, upload both files as a **private Kaggle Dataset**:
   `data/processed/ragtruth_train.jsonl` and `ragtruth_test.jsonl` (about 64 MB together).
   Name it `ragtruth-processed`. Add it to this notebook: **File -> Add Data -> Your Datasets**.
   Then confirm the path in the DATA_DIR cell - Kaggle mounts it at
   `/kaggle/input/<dataset-slug>/`.
2. **Add secrets.** Add-ons -> Secrets:
   - `GITHUB_TOKEN` - a fine-grained PAT with read access to the repo. Required while
     the repo is private.
   - `WANDB_API_KEY` - optional. Without it training runs fine and logs nothing.
   - `HF_TOKEN` - optional. ModernBERT-base is public; a token only raises rate limits.
3. **Turn on the GPU.** Settings -> Accelerator -> GPU P100 (or T4 x2).
4. **Turn on internet.** Needed for the clone, pip and the ModernBERT download.
5. **Save & Run All (Commit)**, not interactive run. It executes in the background,
   survives the browser closing, and keeps the output. This is the whole point of
   using Kaggle.

Watch the first evaluation block. If `answers truncated` is anything but 0, stop -
`max_length` is cutting labels off the end of answers and every number after that
is wrong.

In [ ]:
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        print(f"secret {name} not set")
        return None


GITHUB_TOKEN = secret("GITHUB_TOKEN")
for name in ("WANDB_API_KEY", "HF_TOKEN"):
    value = secret(name)
    if value:
        os.environ[name] = value

REPO_DIR = "/kaggle/working/trustrag"
url = (
    f"https://{GITHUB_TOKEN}@github.com/Wimukthi316/TrustRAG.git"
    if GITHUB_TOKEN
    else "https://github.com/Wimukthi316/TrustRAG.git"
)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", url, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle ships torch already; reinstalling it wastes minutes and risks a CUDA
# mismatch. Only the pieces Kaggle is missing or has an old version of.
!pip install -q -U "transformers>=5.0" pyyaml wandb

# seqeval is published as a source distribution only, and its setup.py fails to
# build under pip's isolated build environment on Kaggle. Installing it in its
# own call keeps that failure from aborting the line above, and --no-build-isolation
# lets it use the setuptools already in the image. It is optional: evaluate_c1
# catches the missing import and reports every metric except span_seqeval.
!pip install -q --no-build-isolation seqeval || echo "seqeval unavailable - span_seqeval will be skipped"

import torch
import transformers

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
if not torch.cuda.is_available():
    raise SystemExit("no GPU - set Accelerator to GPU before running")

name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
major, minor = torch.cuda.get_device_capability(0)
print(f"gpu {name} ({total:.1f} GB) | sm_{major}{minor} | autocast {'bf16' if major >= 8 else 'fp16'}")

# torch.cuda.is_available() is not enough. Kaggle's torch build dropped Pascal:
# on a P100 (sm_60) it reports cuda True and then every kernel launch fails with
# "no kernel image is available for execution on the device". Check the compute
# capability against what this build was compiled for, and stop here if the GPU
# is not in the list, rather than an hour into training.
supported = torch.cuda.get_arch_list()
print("torch was built for:", " ".join(supported))
if f"sm_{major}{minor}" not in supported:
    raise SystemExit(
        f"this torch build does not support sm_{major}{minor} ({name}). "
        "Switch Accelerator to GPU T4 x2 and restart the session."
    )

# Not a guess: a 1-second matmul is the only way to know the kernels really run.
probe = (torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")).sum()
torch.cuda.synchronize()
print("gpu matmul probe ok:", float(probe) == float(probe))

In [ ]:
# Point the repo's relative data paths at the attached Kaggle Dataset. Symlinks
# rather than copies: 64 MB copied twice is 64 MB of session disk for nothing.
import glob
from pathlib import Path

# Kaggle does not always mount a dataset at /kaggle/input/<slug>; the layout has
# changed before and a nested path is easy to get wrong by hand. Find the file
# instead of assuming where it is, and print what was actually there when it fails.
found = glob.glob("/kaggle/input/**/ragtruth_train.jsonl", recursive=True)
if not found:
    print("nothing matched. what is mounted under /kaggle/input:")
    for path in sorted(glob.glob("/kaggle/input/**", recursive=True))[:60]:
        print(" ", path)
    raise SystemExit(
        "ragtruth_train.jsonl not found - attach the ragtruth-processed dataset "
        "via Add Input, then re-run this cell"
    )

DATA_DIR = os.path.dirname(found[0])
print("DATA_DIR", DATA_DIR)

processed = Path(REPO_DIR) / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

for name in ("ragtruth_train.jsonl", "ragtruth_test.jsonl"):
    source = Path(DATA_DIR) / name
    if not source.exists():
        raise SystemExit(f"{source} missing from the attached dataset")
    target = processed / name
    if not target.exists():
        target.symlink_to(source)
    lines = sum(1 for _ in source.open(encoding="utf-8"))
    print(f"{name}: {lines:,} records")

# Expected: 15,090 train and 2,700 test, from the verified preprocessing run.
# Different numbers mean a different build_examples flag combination was used.

In [ ]:
# One quick pass to prove the data path works before spending GPU hours on it.
!python -m src.c1_detector.train_c1 \
    --config configs/c1_smoke.yaml \
    --limit 200 \
    --out-dir /kaggle/working/results/c1/kaggle-smoke \
    --run-name c1-kaggle-smoke \
    --no-wandb

In [ ]:
# Which run this notebook executes. The defaults reproduce the reported run.
# For an ablation change all three together and nothing else, e.g.
#     CONFIG, RUN, TAG = "configs/c1_cw3.yaml", "modernbert-base-cw3", "-cw3"
# TAG keeps the evaluation outputs in their own directories, so an ablation can
# never overwrite the artifacts a reported number was computed from.
CONFIG = "configs/c1_base.yaml"
RUN = "modernbert-base"
TAG = ""
print("config", CONFIG, "| run dir", RUN, "| eval tag", TAG or "(none)")

In [ ]:
# The real run. Everything about it is in the config file, which is committed,
# so this result can be reconstructed later. If it OOMs, halve train.batch_size and
# double train.grad_accum in that file - the effective batch stays 8 and the run
# stays comparable to earlier ones.
!python -m src.c1_detector.train_c1 \
    --config {CONFIG} \
    --out-dir /kaggle/working/results/c1/{RUN}

In [ ]:
# Held-out test set. These are the numbers that go in the report - the per-task
# breakdown, and the example-level F1 that compares against LettuceDetect's 79.22%.
# --dump-probs writes the per-span probabilities C2 calibrates on.
!python -m src.c1_detector.evaluate_c1 \
    --checkpoint /kaggle/working/results/c1/{RUN}/best \
    --data data/processed/ragtruth_test.jsonl \
    --max-length 3072 \
    --batch-size 8 \
    --num-workers 2 \
    --dump-probs \
    --out-dir /kaggle/working/results/c1/test{TAG}

In [ ]:
# Same evaluation over the calibration split, which the model never trained on and
# never had epochs selected on. This file is C2's input: split conformal needs
# scores from data the detector has not seen, or the coverage guarantee is void.
import json

split_ids = json.load(
    open(f"/kaggle/working/results/c1/{RUN}/split_ids.json", encoding="utf-8")
)
calib_ids = set(split_ids["calib"])
print(f"calibration split: {len(calib_ids):,} responses")

calib_path = "/kaggle/working/ragtruth_calib.jsonl"
kept = 0
with open(calib_path, "w", encoding="utf-8") as out:
    for line in open("data/processed/ragtruth_train.jsonl", encoding="utf-8"):
        if json.loads(line)["id"] in calib_ids:
            out.write(line)
            kept += 1
print(f"wrote {kept:,} records to {calib_path}")
assert kept == len(calib_ids), "calibration ids did not all resolve to records"

In [ ]:
!python -m src.c1_detector.evaluate_c1 \
    --checkpoint /kaggle/working/results/c1/{RUN}/best \
    --data /kaggle/working/ragtruth_calib.jsonl \
    --max-length 3072 \
    --batch-size 8 \
    --num-workers 2 \
    --dump-probs \
    --out-dir /kaggle/working/results/c1/calib{TAG}

In [ ]:
# What to download from the committed notebook's Output tab.
#
#   results/c1/test/metrics.json           the reportable numbers
#   results/c1/test/probabilities.jsonl    C2's test-side input
#   results/c1/calib/probabilities.jsonl   C2's calibration-side input
#   results/c1/modernbert-base/summary.json + split_ids.json + history.json
#   results/c1/modernbert-base/best/       the checkpoint
#
# The clone is deleted so it does not end up in the notebook output; it is already
# on GitHub.
import shutil

for path, _, files in os.walk("/kaggle/working/results"):
    for name in sorted(files):
        full = os.path.join(path, name)
        print(f"{os.path.getsize(full)/1e6:8.2f} MB  {full}")

os.chdir("/kaggle/working")
shutil.rmtree(REPO_DIR, ignore_errors=True)